In [9]:
try:
    # prefer parent path so notebook can be run from this folder
    %pip install -r ../../../requirements.txt
except Exception:
    # fallback to local path
    %pip install -r requirements.txt

print("Installation command executed. Restart kernel if necessary.")

Note: you may need to restart the kernel to use updated packages.
Installation command executed. Restart kernel if necessary.


# Deploy a TensorFlow model served with TF Serving using a custom container in an online endpoint
Learn how to deploy a custom container as an online endpoint in Azure Machine Learning.

Custom container deployments can use web servers other than the default Python Flask server used by Azure Machine Learning. Users of these deployments can still take advantage of Azure Machine Learning's built-in monitoring, scaling, alerting, and authentication.

## Prerequisites

* To use Azure Machine Learning, you must have an Azure subscription. If you don't have an Azure subscription, create a free account before you begin. Try the [free or paid version of Azure Machine Learning](https://azure.microsoft.com/free/).

* Install and configure the [Python SDK v2](sdk/setup.sh).

* You must have an Azure resource group, and you (or the service principal you use) must have Contributor access to it.

* You must have an Azure Machine Learning workspace. 

* To deploy locally, you must install [Docker Engine](https://docs.docker.com/engine/install/) on your local computer. We highly recommend this option, so it's easier to debug issues.

# 1. Connect to Azure Machine Learning Workspace

The [workspace](https://docs.microsoft.com/en-us/azure/machine-learning/concept-workspace) is the top-level resource for Azure Machine Learning, providing a centralized place to work with all the artifacts you create when you use Azure Machine Learning. In this section we will connect to the workspace in which the job will be run.

## 1.1. Import the required libraries

In [17]:
from azure.ai.ml import MLClient

from azure.ai.ml.entities import (

    Environment,

    ManagedOnlineDeployment,

    ManagedOnlineEndpoint,

    Model,

)

from azure.identity import AzureCliCredential


## 1.2. Configure workspace details and get a handle to the workspace

To connect to a workspace, we need identifier parameters - a subscription, resource group and workspace name. We will use these details in the `MLClient` from `azure.ai.ml` to get a handle to the required Azure Machine Learning workspace. We use the default [default azure authentication](https://docs.microsoft.com/en-us/python/api/azure-identity/azure.identity.defaultazurecredential?view=azure-python) for this tutorial. Check the [configuration notebook](../../jobs/configuration.ipynb) for more details on how to configure credentials and connect to a workspace.

In [3]:
import os

from pathlib import Path



# Find the repository .env whether the kernel starts at the repo root or notebook folder.

env_path = next(

    (parent / ".env" for parent in (Path.cwd(), *Path.cwd().parents) if (parent / ".env").exists()),

    None,

)

if env_path is not None:

    for line in env_path.read_text(encoding="utf-8").splitlines():

        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:

            continue

        key, value = line.split("=", 1)

        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))



subscription_id = os.environ.get("SUBSCRIPTION_ID", "5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25")

resource_group = os.environ.get("DEV_RESOURCE_GROUP") or os.environ.get("RESOURCE_GROUP", "")

workspace = os.environ.get("DEV_WORKSPACE_NAME") or os.environ.get("WORKSPACE_NAME", "")



if not resource_group or not workspace:

    raise ValueError("Set DEV_RESOURCE_GROUP and DEV_WORKSPACE_NAME in the repository .env file.")



print("Loaded workspace configuration:")

print("  SUBSCRIPTION_ID=", subscription_id)

print("  RESOURCE_GROUP=", resource_group)

print("  WORKSPACE_NAME=", workspace)


Loaded workspace configuration:
  SUBSCRIPTION_ID= 5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25
  RESOURCE_GROUP= rg-aml-ws-dev-cc-01
  WORKSPACE_NAME= mlwdevcc01


In [18]:
credential = AzureCliCredential(process_timeout=60)

ml_client = MLClient(

    credential,

    subscription_id,

    resource_group,

    workspace,

)



workspace_details = ml_client.workspaces.get(workspace)

print("Connected to:", workspace_details.name, "in", workspace_details.location)


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected to: mlwdevcc01 in canadacentral


# 2. Download a TensorFlow model

BASE_PATH=endpoints/online/custom-container

AML_MODEL_NAME=tfserving-mounted

MODEL_NAME=half_plus_two

MODEL_BASE_PATH=/var/azureml-app/azureml-models/$AML_MODEL_NAME/1

Download and unzip a model that divides an input by two and adds 2 to the result

`wget https://aka.ms/half_plus_two-model -O $BASE_PATH/half_plus_two.tar.gz`

`tar -xvf $BASE_PATH/half_plus_two.tar.gz -C $BASE_PATH`

In in this sample, we have already downloaded the model.

In [7]:
# Prepare TensorFlow Serving directory structure and register the model
from pathlib import Path
import shutil
from azure.ai.ml.entities import Model

original_model_dir = Path("half_plus_two")
serving_root = Path("tfservingcustom_serving")
version_subdir = serving_root / "half_plus_two" / "1"

if serving_root.exists():
    shutil.rmtree(serving_root)
version_subdir.mkdir(parents=True, exist_ok=True)

for item in original_model_dir.iterdir():
    destination = version_subdir / item.name
    if item.is_dir():
        shutil.copytree(item, destination)
    else:
        shutil.copy2(item, destination)

registered_model = ml_client.models.create_or_update(
    Model(
        name="tfservingcustom",
        path=str(serving_root),
        type="custom_model",
        description="TensorFlow SavedModel packaged for TF Serving",
        tags={"framework": "tensorflow", "format": "saved_model"},
    )
)

print(f"Registered model: {registered_model.name} (version {registered_model.version})")

Uploading tfservingcustom_serving (0.02 MBs): 100%|##########| 23847/23847 [00:00<00:00, 48873.66it/s]




Registered model: tfservingcustom (version 1)


# 3. Test the pinned GPU image locally



Pull and run the exact TensorFlow Serving GPU image used by the Azure ML deployment. Docker Desktop must expose an NVIDIA GPU to Linux containers.


In [1]:
# Pull and start the same pinned GPU image used by Azure ML

import json
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen

TFSERVING_GPU_IMAGE = "docker.io/tensorflow/serving:2.20.0-gpu"
MODEL_NAME = "half_plus_two"
LOCAL_CONTAINER_NAME = "tfserving-gpu-test"

model_root_candidates = [
    Path.cwd() / "tfservingcustom_serving",
    Path.cwd() / "notebooks/deployments/online/custom_image/tfservingcustom_serving",
]

local_model_root = next((path.resolve() for path in model_root_candidates if path.exists()), None)
if local_model_root is None:
    raise FileNotFoundError("Run step 2 first to create tfservingcustom_serving.")
subprocess.run(["docker", "pull", TFSERVING_GPU_IMAGE], check=True)
subprocess.run(
    ["docker", "rm", "--force", LOCAL_CONTAINER_NAME],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

container_id = subprocess.run(
    [
        "docker", "run", "--rm", "--detach", "--gpus", "all",
        "--name", LOCAL_CONTAINER_NAME,
        "--publish", "8501:8501",
        "--mount", f"type=bind,source={local_model_root},target=/models,readonly",
        "--env", f"MODEL_NAME={MODEL_NAME}",
        TFSERVING_GPU_IMAGE,
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()


health_url = f"http://localhost:8501/v1/models/{MODEL_NAME}"

for attempt in range(30):
    try:
        with urlopen(health_url, timeout=5) as response:
            health = json.load(response)
        break
    except Exception
        if attempt == 29:
            logs = subprocess.run(
                ["docker", "logs", LOCAL_CONTAINER_NAME],
                capture_output=True,
                text=True,
            )
            raise RuntimeError(f"TensorFlow Serving did not become ready.\n{logs.stderr}")
        time.sleep(2)

print("Container:", container_id[:12])
print("Image:", TFSERVING_GPU_IMAGE)
print("Model health:", health)


Container: 24f04d158808
Image: docker.io/tensorflow/serving:2.20.0-gpu
Model health: {'model_version_status': [{'version': '1', 'state': 'AVAILABLE', 'status': {'error_code': 'OK', 'error_message': ''}}]}


## 3.2 Verify health



The startup cell waits until TensorFlow Serving reports model version 1 as `AVAILABLE`.


## 3.3 Invoke the model and verify GPU discovery



Send the sample request and inspect the server logs for the NVIDIA device used by TensorFlow Serving.


In [ ]:
from urllib.request import Request, urlopen

request_path_candidates = [
    Path.cwd() / "sample-request.json",
    Path.cwd() / "notebooks/deployments/online/custom_image/sample-request.json",
]

request_path = next((path for path in request_path_candidates if path.exists()), None)
if request_path is None:
    raise FileNotFoundError("sample-request.json was not found.")

prediction_request = Request(
    f"http://localhost:8501/v1/models/{MODEL_NAME}:predict",
    data=request_path.read_bytes(),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(prediction_request, timeout=30) as response:
    prediction = json.load(response)

container_logs = subprocess.run(
    ["docker", "logs", LOCAL_CONTAINER_NAME],
    check=True,
    capture_output=True,
    text=True,
)

combined_logs = container_logs.stdout + container_logs.stderr

gpu_log_lines = [
    line for line in combined_logs.splitlines()
    if "Created device" in line or "Successfully loaded servable" in line
]

if "device:GPU:0" not in combined_logs:
    raise RuntimeError("TensorFlow Serving loaded the model but did not report GPU:0.")

print("Prediction:", prediction)
print("GPU/model evidence:")
print("\n".join(gpu_log_lines))


Prediction: {'predictions': [2.5, 3.0, 4.5]}
GPU/model evidence:
I0000 00:00:1784732390.971171      41 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1754 MB memory:  -> device: 0, name: NVIDIA RTX A2000 Laptop GPU, pci bus id: 0000:f3:00.0, compute capability: 8.6
2026-07-22 14:59:51.859245: I tensorflow_serving/core/loader_harness.cc:104] Successfully loaded servable version {name: half_plus_two version: 1}


## 3.4 Optional cleanup



The local container remains available at `http://localhost:8501` for demonstrations. Stop it when the demo is finished:



```powershell

docker stop tfserving-gpu-test

```


# 4. Deploy your online endpoint to Azure
Next, deploy your online endpoint to Azure.

## 4.1 Configure online endpoint
`endpoint_name`: The name of the endpoint. It must be unique in the Azure region. Naming rules are defined under [managed online endpoint limits](https://docs.microsoft.com/azure/machine-learning/how-to-manage-quotas#azure-machine-learning-managed-online-endpoints-preview).

`auth_mode` : Use `key` for key-based authentication. Use `aml_token` for Azure Machine Learning token-based authentication. A `key` does not expire, but `aml_token` does expire. 

Optionally, you can add description, tags to your endpoint.

In [9]:
# Creating a unique endpoint name with current datetime to avoid conflicts
import datetime

online_endpoint_name = "endpoint-gpu" + datetime.datetime.now().strftime("%m%d%H%M%f")

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="this is a sample online endpoint",
    auth_mode="key",
    tags={"foo": "bar"},
)

## 4.2 Create the endpoint
Using the `MLClient` created earlier, we will now create the Endpoint in the workspace. This command will start the endpoint creation and return a confirmation response while the endpoint creation continues.

In [11]:
ml_client.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://endpoint-gpu07221117100005.canadacentral.inference.ml.azure.com/score', 'openapi_uri': 'https://endpoint-gpu07221117100005.canadacentral.inference.ml.azure.com/swagger.json', 'name': 'endpoint-gpu07221117100005', 'description': 'this is a sample online endpoint', 'tags': {'foo': 'bar'}, 'properties': {'createdBy': 'System Administrator', 'createdAt': '2026-07-22T15:18:48.379911+0000', 'lastModifiedAt': '2026-07-22T15:18:48.379911+0000', 'azureml.onlineendpointid': '/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/resourcegroups/rg-aml-ws-dev-cc-01/providers/microsoft.machinelearningservices/workspaces/mlwdevcc01/onlineendpoints/endpoint-gpu07221117100005', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70ff6b25/providers/Microsoft.MachineLearningServices/locations/canadacentral/mfeOperationsStatus/oeidp:62b88fa4-5f4a

## 4.3 Configure online deployment
A deployment is a set of resources required for hosting the model that does the actual inferencing. We will create a deployment for our endpoint using the `ManagedOnlineDeployment` class.

### Key aspects of deployment 
- `name` - Name of the deployment.
- `endpoint_name` - Name of the endpoint to create the deployment under.
- `model` - The model to use for the deployment. This value can be either a reference to an existing versioned model in the workspace or an inline model specification.
- `environment` - The environment to use for the deployment. This value can be either a reference to an existing versioned environment in the workspace or an inline environment specification.
- `code_configuration` - the configuration for the source code and scoring script
    - `path`- Path to the source code directory for scoring the model
    - `scoring_script` - Relative path to the scoring file in the source code directory
- `instance_type` - The VM size to use for the deployment. For the list of supported sizes, see [Managed online endpoints SKU list](https://docs.microsoft.com/en-us/azure/machine-learning/reference-managed-online-endpoints-vm-sku-list).
- `instance_count` - The number of instances to use for the deployment

In [19]:
# Create a GPU deployment backed by the TensorFlow Serving model.

try:

    model = ml_client.models.get(name="tfservingcustom", version=str(registered_model.version))

    model_version = registered_model.version

except NameError:

    model_versions = list(ml_client.models.list(name="tfservingcustom"))

    if not model_versions:

        raise RuntimeError("No registered model named tfservingcustom found. Run step 2 first.")

    model = sorted(model_versions, key=lambda item: int(item.version), reverse=True)[0]

    model_version = model.version



TFSERVING_GPU_IMAGE = globals().get(

    "TFSERVING_GPU_IMAGE", "docker.io/tensorflow/serving:2.20.0-gpu"

)

DEPLOYMENT_NAME = "orange"

GPU_INSTANCE_TYPE = "Standard_NC4as_T4_v3"

azure_model_mount = f"/var/azureml-app/azureml-models/{model.name}/{model_version}"

model_mount_base = f"{azure_model_mount}/tfservingcustom_serving"



env = Environment(

    image=TFSERVING_GPU_IMAGE,

    inference_config={

        "liveness_route": {"port": 8501, "path": "/v1/models/half_plus_two"},

        "readiness_route": {"port": 8501, "path": "/v1/models/half_plus_two"},

        "scoring_route": {"port": 8501, "path": "/v1/models/half_plus_two:predict"},

    },

)



environment_variables = {

    "MODEL_BASE_PATH": model_mount_base,

    "MODEL_NAME": "half_plus_two",

}



gpu_deployment = ManagedOnlineDeployment(

    name=DEPLOYMENT_NAME,

    endpoint_name=online_endpoint_name,

    model=model,

    environment=env,

    environment_variables=environment_variables,

    instance_type=GPU_INSTANCE_TYPE,

    instance_count=1,

)



print("Image:", TFSERVING_GPU_IMAGE)

print("Model:", model.name, "version", model_version)

print("MODEL_BASE_PATH:", model_mount_base)

print("GPU SKU:", GPU_INSTANCE_TYPE)


Image: docker.io/tensorflow/serving:2.20.0-gpu
Model: tfservingcustom version 1
MODEL_BASE_PATH: /var/azureml-app/azureml-models/tfservingcustom/1/tfservingcustom_serving
GPU SKU: Standard_NC4as_T4_v3


In [ ]:
# Inspect packaged model structure before deployment
from pathlib import Path

print("Packaged model directory structure:")
for path in sorted(Path("tfservingcustom_serving").rglob("*")):
    rel = path.relative_to("tfservingcustom_serving")
    if path.is_dir():
        print(f"  [DIR] {rel}")
    else:
        print(f"  [FILE] {rel}")

### Readiness route vs. liveness route
An HTTP server defines paths for both liveness and readiness. A liveness route is used to check whether the server is running. A readiness route is used to check whether the server is ready to do work. In machine learning inference, a server could respond 200 OK to a liveness request before loading a model. The server could respond 200 OK to a readiness request only after the model has been loaded into memory.

Review the [Kubernetes documentation](https://kubernetes.io/docs/tasks/configure-pod-container/configure-liveness-readiness-startup-probes/) for more information about liveness and readiness probes.

Notice that this deployment uses the same path for both liveness and readiness, since TF Serving only defines a liveness route.

## 4.4 Create the deployment
Using the `MLClient` created earlier, we will now create the deployment in the workspace. This command will start the deployment creation and return a confirmation response while the deployment creation continues.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

try:
    existing_deployment = ml_client.online_deployments.get(
        name=DEPLOYMENT_NAME,
        endpoint_name=online_endpoint_name,
    )

    if existing_deployment.provisioning_state == "Failed":
        print(f"Deleting failed deployment {DEPLOYMENT_NAME} before retrying...")
        ml_client.online_deployments.begin_delete(
            name=DEPLOYMENT_NAME,
            endpoint_name=online_endpoint_name,
        ).result()
except ResourceNotFoundError:
    pass

ml_client.begin_create_or_update(gpu_deployment).result()

Check: endpoint endpoint-gpu07221117100005 exists


..............................................................................

In [ ]:
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.begin_create_or_update(endpoint).result()
print("Traffic:", endpoint.traffic)

# 5. Test the endpoint with sample data
Using the `MLClient` created earlier, we will get a handle to the endpoint. The endpoint can be invoked using the `invoke` command with the following parameters:
- `endpoint_name` - Name of the endpoint
- `request_file` - File with request data
- `deployment_name` - Name of the specific deployment to test in an endpoint

We will send a sample request using a [json](./model-1/sample-request.json) file. 

In [ ]:
response = ml_client.online_endpoints.invoke(
    endpoint_name=online_endpoint_name,
    deployment_name=DEPLOYMENT_NAME,
    request_file="sample-request.json",
)
print(response)

'{\n    "predictions": [2.5, 3.0, 4.5\n    ]\n}'

# 6. Managing endpoints and deployments

## 6.1 Get details of the endpoint

In [ ]:
# Get the details for online endpoint
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

# existing traffic details
print(endpoint.traffic)

# Get the scoring URI
print(endpoint.scoring_uri)

## 6.2 Get the TensorFlow Serving deployment logs



Inspect the `orange` deployment logs for model loading, GPU initialization, and request handling.


In [ ]:
logs = ml_client.online_deployments.get_logs(
    name=DEPLOYMENT_NAME,
    endpoint_name=online_endpoint_name,
    lines=100,
)
print(logs)

# 7. Delete the endpoint

In [ ]:
#ml_client.online_endpoints.begin_delete(name=online_endpoint_name)